# HW3: M5 Price Elasticity and Forecasting

Цель:

- выбрать несколько SKU с частыми продажами и как минимум 10 разными ценами;
- оценить ценовую эластичность каждого SKU и решить, подходит ли он для промо;
- сравнить 3 способа прогноза спроса на отложенном горизонте 28 дней.


## План анализа

1. Скачать и подготовить датасет M5.
2. Автоматически выбрать SKU с высокой частотой продаж и богатой ценовой историей.
3. Построить кривую эластичности по каждому SKU.
4. Сравнить три прогноза:
   - без знания будущей цены;
   - с будущей ценой как признаком модели;
   - базовая модель + кривая спроса от цены.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, Markdown, display

from hw3.pipeline import default_config, run_pipeline

PROJECT_DIR = Path.cwd()
CONFIG = default_config(PROJECT_DIR)
TABLES_DIR = CONFIG.tables_dir
FIGURES_DIR = CONFIG.figures_dir


In [ ]:
if not (TABLES_DIR / "analysis_summary.json").exists():
    run_pipeline(CONFIG)

with open(TABLES_DIR / "analysis_summary.json", "r", encoding="utf-8") as file:
    summary_payload = json.load(file)

selected_skus = pd.read_csv(TABLES_DIR / "selected_skus.csv")
elasticity_summary = pd.read_csv(TABLES_DIR / "elasticity_summary.csv")
metrics_by_sku = pd.read_csv(TABLES_DIR / "forecast_metrics_by_sku.csv")
method_summary = pd.read_csv(TABLES_DIR / "forecast_method_summary.csv")
final_recommendations = pd.read_csv(TABLES_DIR / "final_recommendations.csv")
predictions = pd.read_csv(TABLES_DIR / "holdout_predictions.csv", parse_dates=["date"])


## Выбранные SKU

In [ ]:
selected_skus[
    [
        "sku_id",
        "dept_id",
        "cat_id",
        "positive_sales_ratio",
        "total_sales",
        "unique_prices",
        "min_price",
        "median_price",
        "max_price",
    ]
].round(3)


## Методика

- `Модель 1`: градиентный бустинг по лагам, календарю и SNAP, но без цены будущего периода.
- `Модель 2`: тот же тип модели, но с признаками текущей цены.
- `Модель 3`: прогноз `Модели 1`, скорректированный через отдельную монотонную кривую эластичности.

Кривая эластичности оценивается как связь между ценой и мультипликатором спроса относительно базового прогноза без цены.


## Итоги по эластичности

In [ ]:
elasticity_summary.sort_values("local_elasticity")[[
    "sku_id",
    "unique_prices",
    "local_elasticity",
    "promo_lift",
    "recommendation",
]]


## Качество прогноза

In [ ]:
method_summary.assign(
    mae=lambda df: df["mae"].round(3),
    rmse=lambda df: df["rmse"].round(3),
    wape=lambda df: df["wape"].round(3),
    bias=lambda df: df["bias"].round(3),
)


In [ ]:
metrics_by_sku.assign(
    mae=lambda df: df["mae"].round(3),
    rmse=lambda df: df["rmse"].round(3),
    wape=lambda df: df["wape"].round(3),
    bias=lambda df: df["bias"].round(3),
).sort_values(["sku_id", "wape"])


## Выводы по SKU

In [ ]:
for narrative in summary_payload["narratives"]:
    display(Markdown(f"- {narrative}"))


## Визуализации

In [ ]:
for sku_id in selected_skus["sku_id"]:
    display(Markdown(f"### {sku_id}"))
    display(Image(filename=str(FIGURES_DIR / f"elasticity_{sku_id.replace('|', '_')}.png")))
    display(Image(filename=str(FIGURES_DIR / f"forecast_{sku_id.replace('|', '_')}.png")))


## Общий анализ

- Если `Модель 2` лучше, значит будущая цена действительно несёт полезный сигнал и модель смогла его выучить.
- Если сильнее `Модель 3`, значит явная кривая спроса добавляет структуру, которую бустинг сам не вытащил.
- Если выигрывает `Модель 1`, цена либо слабо влияет на спрос, либо влияние нестабильно и тонет в шуме.

Что можно улучшить дальше:

- добавить иерархические признаки по категории и магазину;
- использовать rolling validation для настройки гиперпараметров;
- заменить локальную изотоническую кривую на байесовскую или сплайновую модель с доверительными интервалами;
- учесть каннибализацию, праздники и глубину промо отдельно.
